# Q1 2026 ML Alpha Case Study Notebook

This notebook is the canonical Milestone 23 notebook example for the Q1 2026 case study. It uses the public `src.execution` APIs to run the `ml_cross_sectional_xgb_2026_q1` alpha, inspect persisted artifacts through `ExecutionResult`, and build the pinned alpha-sleeve portfolio view.

The notebook is a documentation and demo artifact. It does not introduce notebook-only execution behavior, hidden persistence, or alternate artifact contracts.

## Prerequisites

Run this notebook from the repository root after installing the normal project dependencies. Jupyter itself is optional and is not a core package dependency.

The end-to-end alpha cells require the real Q1 2026 `features_daily` surface described in `docs/backfilled_2026_q1_alpha_workflow.md` and `docs/ml_cross_sectional_xgb_2026_q1.md`.

The portfolio cell uses `configs/portfolios_alpha_2026_q1.yml`, which is pinned to the documented XGBoost alpha run id `ml_cross_sectional_xgb_2026_q1_alpha_eval_d4c845bd3f5c`. If you rerun the alpha and want the portfolio to consume the fresh run instead, update that config intentionally before running the portfolio cell.

## Notebook Or CLI

Use this notebook for exploration, artifact inspection, and interactive review. Use the CLI for operational runs, automation, CI, release bundles, and workflows where process exit behavior is part of the contract.

Both surfaces preserve the same deterministic artifacts, manifests, registries, metrics, and validation/report schemas.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.execution import run_alpha, run_docs_path_lint, run_portfolio

ALPHA_NAME = "ml_cross_sectional_xgb_2026_q1"
ALPHA_CONFIG = "configs/alphas_2026_q1.yml"
PORTFOLIO_NAME = "ml_cross_sectional_xgb_2026_q1_equal"
PORTFOLIO_CONFIG = "configs/portfolios_alpha_2026_q1.yml"

## 1. Run The Q1 2026 XGBoost Alpha

`run_alpha(..., mode="full")` trains and evaluates the alpha, maps predictions into deterministic sleeve signals, writes sleeve artifacts, updates the alpha registry, and returns an `ExecutionResult` for notebook inspection.

In [ ]:
alpha_result = run_alpha(
    ALPHA_NAME,
    config_path=ALPHA_CONFIG,
    mode="full",
)

alpha_result.notebook_summary()

## 2. Inspect Alpha Artifacts

Inspect named outputs instead of reconstructing filenames. The helper methods below read artifacts that were written by the execution workflow; they do not create notebook-only state.

In [ ]:
alpha_outputs = alpha_result.output_keys()
alpha_metrics = alpha_result.load_metrics_json("alpha_metrics_json")
sleeve_metrics = alpha_result.load_metrics_json("sleeve_metrics_json")
alpha_manifest = alpha_result.load_manifest()

alpha_paths = {
    "manifest": alpha_result.output_path("manifest_json", must_exist=True),
    "predictions": alpha_result.output_path("predictions_parquet", must_exist=True),
    "signals": alpha_result.output_path("signals_parquet", must_exist=True),
    "sleeve_metrics": alpha_result.output_path("sleeve_metrics_json", must_exist=True),
}

{
    "outputs": alpha_outputs,
    "mean_ic": alpha_metrics.get("mean_ic"),
    "ic_ir": alpha_metrics.get("ic_ir"),
    "sleeve_sharpe": sleeve_metrics.get("sharpe_ratio"),
    "promotion_status": alpha_manifest.get("promotion", {}).get("promotion_status"),
    "paths": alpha_paths,
}

## 3. Build The Alpha-Sleeve Portfolio View

The Q1 2026 portfolio config consumes an alpha sleeve through the standard portfolio artifact contract. This cell uses the checked-in config directly, so it is deterministic and auditable. The config is intentionally explicit about the upstream run id it consumes.

In [ ]:
portfolio_result = run_portfolio(
    portfolio_config_path=PORTFOLIO_CONFIG,
    portfolio_name=PORTFOLIO_NAME,
    timeframe="1D",
)

portfolio_result.notebook_summary()

## 4. Inspect Portfolio Metrics And QA

Portfolio artifacts preserve the CLI contract: metrics, weights, returns, equity curve, QA summary, and manifest are all written under the deterministic portfolio artifact directory.

In [ ]:
portfolio_metrics = portfolio_result.load_metrics_json()
portfolio_qa = portfolio_result.load_summary_json("qa_summary_json")
portfolio_manifest = portfolio_result.load_manifest()

{
    "outputs": portfolio_result.output_keys(),
    "total_return": portfolio_metrics.get("total_return"),
    "sharpe_ratio": portfolio_metrics.get("sharpe_ratio"),
    "qa_status": portfolio_qa.get("status"),
    "manifest_run_id": portfolio_manifest.get("run_id"),
    "weights": portfolio_result.output_path("weights_csv", must_exist=True),
    "returns": portfolio_result.output_path("portfolio_returns_csv", must_exist=True),
}

## 5. Inspect A Validation Report From Python

Validation APIs return structured results even when a report status is failed. CLI wrappers may exit non-zero for failed validation reports; notebooks can inspect the report first and decide how to handle it.

In [ ]:
docs_lint_result = run_docs_path_lint(
    output="artifacts/qa/q1_2026_notebook_docs_path_lint.json"
)
docs_lint_report = docs_lint_result.load_metrics_json("report_json")

{
    "summary": docs_lint_result.notebook_summary(),
    "report_path": docs_lint_result.output_path("report_json", must_exist=True),
    "status": docs_lint_report.get("status"),
    "finding_count": docs_lint_report.get("finding_count"),
}

## What This Notebook Demonstrates

- Stable notebook-facing imports from `src.execution`.
- Real Q1 2026 alpha execution using `run_alpha`.
- Downstream alpha-sleeve portfolio construction using `run_portfolio`.
- Validation report inspection using `run_docs_path_lint`.
- Artifact-first inspection through `notebook_summary()`, `output_keys()`, `output_path(...)`, `load_manifest()`, `load_metrics_json()`, and `load_summary_json()`.

The notebook remains a guided example with documented prerequisites. It is fully runnable when the Q1 2026 `features_daily` data and pinned alpha-sleeve artifacts are available in the local repository environment.